In [ ]:
!pip install -q anthropic

from google.colab import drive
drive.mount("/content/drive")

import json
import re
import time
import hashlib
import getpass
import numpy as np
import pandas as pd

from pathlib import Path
from IPython.display import display

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 24.5 MB/s eta 0:00:00
Mounted at /content/drive


In [ ]:
import anthropic


ANTHROPIC_API_KEY = ""

ANTHROPIC_API_KEY = ANTHROPIC_API_KEY.strip()

assert ANTHROPIC_API_KEY.startswith("sk-ant-"), (
    "Invalid Anthropic API key format."
)

claude_client = anthropic.Anthropic(
    api_key=ANTHROPIC_API_KEY
)

print("Anthropic client object created.")

Anthropic client object created.


In [ ]:
RESULT_DIR = Path(
    "/content/drive/MyDrive/cbt_results"
)

INPUT_PATH = (
    RESULT_DIR / "responses_for_judge.json"
)


OUTPUT_PATH = (
    RESULT_DIR / "ft_rag_judge_temperature0.json"
)

assert INPUT_PATH.exists(), (
    f"Input file not found: {INPUT_PATH}"
)

with open(INPUT_PATH, "r", encoding="utf-8") as file:
    single_turn_cases = json.load(file)

assert isinstance(single_turn_cases, list)
assert len(single_turn_cases) == 100, (
    f"Expected 100 cases, found "
    f"{len(single_turn_cases)}."
)

seen_ids = set()

for position, case in enumerate(single_turn_cases):
    assert "id" in case, (
        f"Missing id in record {position}."
    )

    case_id = str(case["id"])

    assert case_id not in seen_ids, (
        f"Duplicate case id: {case_id}"
    )
    seen_ids.add(case_id)

    assert isinstance(
        case.get("user_input"),
        str,
    ) and case["user_input"].strip(), (
        f"Missing user_input for case {case_id}."
    )

    assert isinstance(
        case.get("ft_rag_response"),
        str,
    ) and case["ft_rag_response"].strip(), (
        f"Missing ft_rag_response for case {case_id}."
    )

print(f"Loaded {len(single_turn_cases)} cases.")
print("Input file:", INPUT_PATH)
print("New output file:", OUTPUT_PATH)
print("The input file will remain unchanged.")

Loaded 100 cases.
Input file: /content/drive/MyDrive/cbt_results/responses_for_judge.json
New output file: /content/drive/MyDrive/cbt_results/ft_rag_judge_temperature0.json
The input file will remain unchanged.


In [ ]:
JUDGE_MODEL = "claude-sonnet-4-5"
JUDGE_TEMPERATURE = 0
JUDGE_MAX_TOKENS = 350
PROMPT_VERSION = "ft_rag_single_turn_v1"

JUDGE_DIMS = [
    "CBT_Accuracy",
    "Empathy",
    "Relevance",
    "Clinical_Appropriateness",
    "Overall",
]

KEY_ALIASES = {
    "CBT_Accuracy": [
        "CBT_Accuracy",
        "CBT Accuracy",
        "CBTAccuracy",
        "cbt_accuracy",
    ],
    "Empathy": [
        "Empathy",
        "empathy",
    ],
    "Relevance": [
        "Relevance",
        "relevance",
    ],
    "Clinical_Appropriateness": [
        "Clinical_Appropriateness",
        "Clinical Appropriateness",
        "ClinicalAppropriateness",
        "clinical_appropriateness",
    ],
    "Overall": [
        "Overall",
        "overall",
        "Average",
        "average",
    ],
}

JUDGE_PROMPT_TEMPLATE = """
You are an expert evaluator of CBT
(Cognitive Behavioral Therapy) chatbot responses.

Evaluate the following response to a client's message
on these 4 dimensions, scoring each from 1-10:

1. CBT_Accuracy: Does the response correctly apply CBT
techniques such as Socratic questioning, cognitive
restructuring, thought challenging, identifying automatic
thoughts, or behavioral activation?

2. Empathy: Is the response warm, non-judgmental,
validating, and emotionally attuned?

3. Relevance: Does the response directly address the
client's message without going off-topic?

4. Clinical_Appropriateness: Is the response professionally
appropriate for a CBT support context, avoiding diagnosis,
overclaiming, unsafe advice, or replacing professional care?

Client message:
{user_input}

Response to evaluate:
{response}

Return ONLY valid JSON.
Use EXACTLY these keys:

{{
  "CBT_Accuracy": <number from 1 to 10>,
  "Empathy": <number from 1 to 10>,
  "Relevance": <number from 1 to 10>,
  "Clinical_Appropriateness": <number from 1 to 10>,
  "Overall": <average of the four numeric scores>,
  "Reasoning": "<one concise sentence>"
}}
""".strip()

print("Judge model:", JUDGE_MODEL)
print("Temperature:", JUDGE_TEMPERATURE)
print("Prompt version:", PROMPT_VERSION)

Judge model: claude-sonnet-4-5
Temperature: 0
Prompt version: ft_rag_single_turn_v1


In [ ]:
def response_hash(response):
    return hashlib.sha256(
        response.encode("utf-8")
    ).hexdigest()


def extract_json(raw_text):
    cleaned = (
        raw_text
        .strip()
        .replace("```json", "")
        .replace("```JSON", "")
        .replace("```", "")
        .strip()
    )

    match = re.search(
        r"\{.*\}",
        cleaned,
        re.DOTALL,
    )

    if not match:
        raise ValueError(
            "No JSON object found in response: "
            f"{cleaned[:300]}"
        )

    return json.loads(match.group())


def normalise_judge_scores(parsed):
    if not isinstance(parsed, dict):
        raise ValueError(
            "Judge output is not a JSON dictionary."
        )

    normalised = {}

    for target_key, aliases in KEY_ALIASES.items():
        value = None

        for alias in aliases:
            if alias in parsed:
                value = parsed[alias]
                break

        if value is None:
            # Overall may be calculated from the four dimensions
            if target_key == "Overall":
                continue

            raise ValueError(
                f"Missing judge field: {target_key}"
            )

        try:
            normalised[target_key] = float(value)
        except (TypeError, ValueError):
            raise ValueError(
                f"Invalid value for {target_key}: {value!r}"
            )

    required_four = [
        "CBT_Accuracy",
        "Empathy",
        "Relevance",
        "Clinical_Appropriateness",
    ]

    for dimension in required_four:
        score = normalised[dimension]

        if not 1.0 <= score <= 10.0:
            raise ValueError(
                f"Invalid {dimension} score: {score}"
            )

    if "Overall" not in normalised:
        normalised["Overall"] = float(
            np.mean([
                normalised[dimension]
                for dimension in required_four
            ])
        )

    if not 1.0 <= normalised["Overall"] <= 10.0:
        raise ValueError(
            "Invalid Overall score: "
            f"{normalised['Overall']}"
        )

    normalised["Reasoning"] = (
        parsed.get("Reasoning")
        or parsed.get("reasoning")
        or ""
    )

    return normalised


def judge_ft_rag_response(
    user_input,
    response,
    max_retries=4,
):
    prompt = JUDGE_PROMPT_TEMPLATE.format(
        user_input=user_input,
        response=response,
    )

    last_error = None

    for attempt in range(1, max_retries + 1):
        try:
            message = claude_client.messages.create(
                model=JUDGE_MODEL,
                max_tokens=JUDGE_MAX_TOKENS,
                temperature=JUDGE_TEMPERATURE,
                messages=[
                    {
                        "role": "user",
                        "content": prompt,
                    }
                ],
            )

            raw_text = message.content[0].text
            parsed = extract_json(raw_text)

            return normalise_judge_scores(parsed)

        except Exception as error:
            last_error = error

            print(
                f"    Attempt {attempt}/{max_retries} "
                f"failed: {error}"
            )

            if attempt < max_retries:
                time.sleep(2 + attempt * 2)

    # 不使用默认 5 分，避免无效结果混入数据
    raise RuntimeError(
        "Judge failed after all retries. "
        f"Last error: {last_error}"
    )

In [ ]:
case_order = {
    str(case["id"]): position
    for position, case in enumerate(single_turn_cases)
}

case_by_id = {
    str(case["id"]): case
    for case in single_turn_cases
}


def validate_saved_record(record):
    assert isinstance(record, dict)
    assert "id" in record

    record_id = str(record["id"])

    assert record_id in case_by_id, (
        f"Unknown saved id: {record_id}"
    )

    assert record.get("judge_model") == JUDGE_MODEL, (
        f"Unexpected judge model for id={record_id}."
    )

    assert float(record.get("temperature")) == 0.0, (
        f"Unexpected temperature for id={record_id}."
    )

    assert record.get("prompt_version") == PROMPT_VERSION, (
        f"Unexpected prompt version for id={record_id}."
    )

    expected_hash = response_hash(
        case_by_id[record_id]["ft_rag_response"]
    )

    assert record.get(
        "ft_rag_response_sha256"
    ) == expected_hash, (
        f"FT+RAG response changed for id={record_id}."
    )

    assert "ft_rag_judge_scores" in record

    scores = record["ft_rag_judge_scores"]

    for dimension in JUDGE_DIMS:
        assert dimension in scores, (
            f"Missing {dimension} for id={record_id}."
        )

        value = float(scores[dimension])

        assert np.isfinite(value)
        assert 1.0 <= value <= 10.0


def save_checkpoint(records):
    sorted_records = sorted(
        records,
        key=lambda record: case_order[
            str(record["id"])
        ],
    )

    temporary_path = OUTPUT_PATH.with_suffix(
        ".tmp"
    )

    with open(
        temporary_path,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            sorted_records,
            file,
            ensure_ascii=False,
            indent=2,
        )

    temporary_path.replace(OUTPUT_PATH)

In [ ]:
if OUTPUT_PATH.exists():
    with open(
        OUTPUT_PATH,
        "r",
        encoding="utf-8",
    ) as file:
        ft_rag_temperature0_results = json.load(file)

    assert isinstance(
        ft_rag_temperature0_results,
        list,
    )

    saved_ids = set()

    for record in ft_rag_temperature0_results:
        validate_saved_record(record)

        record_id = str(record["id"])

        assert record_id not in saved_ids, (
            f"Duplicate saved id: {record_id}"
        )

        saved_ids.add(record_id)

    print(
        "Resuming from "
        f"{len(ft_rag_temperature0_results)} "
        "completed cases."
    )
else:
    ft_rag_temperature0_results = []
    saved_ids = set()

for index, case in enumerate(
    single_turn_cases,
    start=1,
):
    case_id = str(case["id"])

    if case_id in saved_ids:
        print(
            f"[{index:03d}/100] "
            f"Case {case_id} already completed."
        )
        continue

    scores = judge_ft_rag_response(
        user_input=case["user_input"],
        response=case["ft_rag_response"],
    )

    result_record = {
        "id": case["id"],
        "category": case.get("category", ""),
        "judge_model": JUDGE_MODEL,
        "temperature": JUDGE_TEMPERATURE,
        "max_tokens": JUDGE_MAX_TOKENS,
        "prompt_version": PROMPT_VERSION,
        "ft_rag_response_sha256": response_hash(
            case["ft_rag_response"]
        ),
        "ft_rag_judge_scores": scores,
    }

    validate_saved_record(result_record)

    ft_rag_temperature0_results.append(
        result_record
    )
    saved_ids.add(case_id)


    save_checkpoint(
        ft_rag_temperature0_results
    )

    print(
        f"[{index:03d}/100] "
        f"{case.get('category', ''):24s} "
        f"Overall={scores['Overall']:.2f}"
    )

    time.sleep(0.3)

assert len(ft_rag_temperature0_results) == 100
assert len(saved_ids) == 100

save_checkpoint(ft_rag_temperature0_results)

print("\nFT+RAG temperature-zero evaluation completed.")
print("New result file:", OUTPUT_PATH)
print("Original result files were not modified.")

[001/100] anxiety                  Overall=9.00
[002/100] anxiety                  Overall=9.00
[003/100] anxiety                  Overall=9.25
[004/100] anxiety                  Overall=8.00
[005/100] anxiety                  Overall=8.00
[006/100] negative_self_talk       Overall=9.25
[007/100] negative_self_talk       Overall=9.25
[008/100] negative_self_talk       Overall=9.50
[009/100] negative_self_talk       Overall=9.50
[010/100] negative_self_talk       Overall=9.50
[011/100] perfectionism            Overall=9.25
[012/100] perfectionism            Overall=8.75
[013/100] perfectionism            Overall=9.25
[014/100] perfectionism            Overall=8.25
[015/100] perfectionism            Overall=9.25
[016/100] catastrophizing          Overall=9.25
[017/100] catastrophizing          Overall=9.00
[018/100] catastrophizing          Overall=9.25
[019/100] catastrophizing          Overall=9.25
[020/100] catastrophizing          Overall=9.25
[021/100] avoidance                Overa

In [ ]:
with open(
    OUTPUT_PATH,
    "r",
    encoding="utf-8",
) as file:
    final_results = json.load(file)

assert len(final_results) == 100

final_ids = set()

for record in final_results:
    validate_saved_record(record)

    record_id = str(record["id"])

    assert record_id not in final_ids
    final_ids.add(record_id)

assert final_ids == set(case_by_id.keys())

summary_rows = []

for dimension in JUDGE_DIMS:
    values = np.asarray([
        float(
            record["ft_rag_judge_scores"][
                dimension
            ]
        )
        for record in final_results
    ])

    summary_rows.append({
        "Dimension": dimension,
        "mean": values.mean(),
        "std": values.std(ddof=1),
        "minimum": values.min(),
        "maximum": values.max(),
    })

summary_df = (
    pd.DataFrame(summary_rows)
    .set_index("Dimension")
)

display(summary_df.round(3))

print(
    "PASS: 100 unique FT+RAG responses were "
    "evaluated at temperature=0."
)
print("Final file:", OUTPUT_PATH)

,mean,std,minimum,maximum
Dimension,,,,
CBT_Accuracy,7.91,1.120,4.00,9.0
Empathy,8.38,0.648,7.00,9.0
Relevance,9.40,1.119,5.00,10.0
Clinical_Appropriateness,9.39,0.723,6.00,10.0
Overall,8.77,0.768,6.25,9.5


PASS: 100 unique FT+RAG responses were evaluated at temperature=0.
Final file: /content/drive/MyDrive/cbt_results/ft_rag_judge_temperature0.json


In [ ]:
four_dimensions = [
    "CBT_Accuracy",
    "Empathy",
    "Relevance",
    "Clinical_Appropriateness",
]

differences = []

for record in final_results:
    scores = record["ft_rag_judge_scores"]

    calculated_overall = np.mean([
        float(scores[dimension])
        for dimension in four_dimensions
    ])

    reported_overall = float(scores["Overall"])

    differences.append(
        abs(reported_overall - calculated_overall)
    )

print(
    "Maximum difference between reported Overall "
    "and the four-dimension average:",
    max(differences),
)

if max(differences) <= 0.01:
    print(
        "PASS: Overall scores match the arithmetic "
        "mean of the four dimensions."
    )
else:
    print(
        "NOTE: Some judge-reported Overall scores "
        "differ from the exact arithmetic mean."
    )

Maximum difference between reported Overall and the four-dimension average: 0.0
PASS: Overall scores match the arithmetic mean of the four dimensions.
